In [ ]:
! pip install -q -U langchain langchain-community langchain-core
! pip install -q -U langchain-google-genai
! pip install -q -U langchain-qdrant langchain-huggingface qdrant-client
! pip install -q -U langchain-pinecone pinecone-notebooks "langchain-chroma>=0.1.2"
! pip install -qU pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 628.3/628.3 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.8/94.8 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.6/278.6 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.8/94.8 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.8/70.8 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.2/73.2 kB 7.5 MB/s eta 0:00:00


## **Document Loading**


In [ ]:
# PDF Loader

from langchain_community.document_loaders import PyPDFLoader

folder_path = "/content/docs/Generative AI Foundations in Python.pdf"
loader = PyPDFLoader(folder_path)
pages = loader.load()
print(len(pages))

194


In [ ]:
# Splits pages into chunks

from langchain.text_splitter import RecursiveCharacterTextSplitter

r_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""]
)

docs = r_splitter.split_documents(pages)
print(len(docs))

566


In [ ]:
# Create embeddings using google embedding model

from google.colab import userdata
from langchain_google_genai import GoogleGenerativeAIEmbeddings

google_api_key = userdata.get('GOOGLE_API_KEY')

embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001", google_api_key = google_api_key)
document_embeddings = embeddings.embed_documents([doc.page_content for doc in docs[0 : 20]])
print(document_embeddings)

[[0.020741678774356842, -0.020972292870283127, -0.03504716977477074, -0.020582225173711777, 0.10053366422653198, 0.002182195195928216, 0.004565094597637653, 0.008226683363318443, 0.04168606549501419, 0.010706353932619095, 0.005042023491114378, 0.020388172939419746, -0.026354264467954636, 0.017669303342700005, 0.0014742566272616386, -0.05862367898225784, 0.06349079310894012, 0.062114063650369644, -0.000559604843147099, -0.005431409925222397, 0.003859724849462509, 0.017806673422455788, 0.0440605953335762, -0.048596423119306564, -0.029835179448127747, -0.01961793564260006, 0.004889351781457663, -0.042199376970529556, -0.012730546295642853, 0.0005593700334429741, -0.03251402825117111, 0.004646633751690388, -0.03889182582497597, -0.0044446163810789585, 0.008356153964996338, -0.05973254144191742, 0.010083555243909359, 0.02953573316335678, -0.02550550177693367, -0.015750084072351456, 0.010546203702688217, -0.024105452001094818, -0.014944839291274548, 0.002445669611915946, -0.03772009909152984

In [ ]:
# Create DB and add ebbeddings to db

from langchain_chroma import Chroma
from google.colab import userdata
from langchain_google_genai import GoogleGenerativeAIEmbeddings

google_api_key = userdata.get('GOOGLE_API_KEY')
embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001", google_api_key = google_api_key)

vector_store = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory="./chroma_langchain_db"
)

print(vector_store._collection.count())

1132


In [ ]:
# Lets Apply Simelarity Search

question = "What is Generative AI?"
docs = vector_store.similarity_search(question, k=3)
print(docs[0].page_content)

Generative methods learn complex relationships through expansive training in order to generate new
data sequences enabling many downstream applications. Effectively, these models create synthetic
outputs by replicating the statistical patterns and properties discovered in training data, capturing
nuances and idiosyncrasies that closely reflect human behaviors.
In practice, a discriminative image classifier labels images containing a cat or a dog. In contrast, a
generative model can synthesize diverse, realistic cat or dog images by learning the distributions of
pixels and implicit features from existing images. Moreover, generative models can be trained across
modalities to unlock new possibilities in synthesis-focused applications to generate human-like
photographs, videos, music, and text.
There are several key methods that have formed the foundation for many of the recent advancements
in Generative AI, each with unique approaches and strengths. In the next section, we survey


In [ ]:
# Creating a Retriever

retriever = vector_store.as_retriever(search_kwargs={"k": 2})
retriever_results = retriever.invoke("Who is the highest-paid F1 driver?")

print(retriever_results)

[Document(id='b76acfc7-df89-4b9d-99a4-05bd88c37685', metadata={'page': 46, 'page_label': '47', 'source': '/content/docs/Generative AI Foundations in Python.pdf', 'text': 'Autoregressive modeling: Autoregressive modeling generates sequences by predicting the next token conditioned only on\nprevious tokens. The model produces outputs one step at a time, with each new token depending on those preceding it.\nAutoregressive transformers such as GPT leverage this technique for controlled text generation.\nTransformers combine self-attention for long-range dependencies, pre-trained representations, and\nautoregressive decoding to adapt to discriminative and generative tasks.\nAdvanced generative synthesis can be achieved with different architectures that make trade-offs\nbetween complexity, scalability, and specialization. For example, instead of using both the encoder\nand decoder, many state-of-the-art generative models employ a decoder-only or encoder-only\napproach. The encoder-decoder fr

In [ ]:
# Create Prompt Template

from langchain_core.prompts import ChatPromptTemplate

template = """Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer. Use three sentences maximum. Keep the answer as concise as possible. Always say "thanks for asking!" at the end of the answer.
{context}
Question: {question}
Helpful Answer:"""

prompt = ChatPromptTemplate.from_template(template)

In [ ]:
# Create a Question Answering Chain

from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.schema.runnable import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


google_api_key = userdata.get('GOOGLE_API_KEY')
llm:ChatGoogleGenerativeAI = ChatGoogleGenerativeAI(api_key=google_api_key, model="gemini-2.0-flash-exp")

# Run chain
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
response = rag_chain.invoke("What is Generative AI?")
print(f"Question: {question}")
print(f"Answer: {response}")

Question: What is Generative AI?
Answer: Generative AI learns complex relationships through extensive training to create new data sequences. These models generate synthetic outputs by replicating the statistical patterns found in training data. They can synthesize diverse, realistic content like images, videos, music, and text. thanks for asking!


In [ ]:
response = rag_chain.invoke("What is Neural Network?")
print(f"Question: {question}")
print(f"Answer: {response}")

Question: What is Generative AI?
Answer: Neural networks use iterative processing and dynamic updating to learn and represent relationships within text. They can capture contextual connections and interdependencies across sentences or entire documents. They also allow information to be passed from one step in a sequence to the next. thanks for asking!
